<a href="https://colab.research.google.com/github/nihar-pu/vehicle-orientation/blob/main/vehicle_and_orientation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install ultralytics
from ultralytics import YOLO

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 52.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.1/78.1 kB 9.4 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install kaggle


In [ ]:
!kaggle.api.authenticate()
!kaggle dataset download -d raghavdharwal/vehicle-orientation-dataset-part-1

/bin/bash: -c: line 2: syntax error: unexpected end of file
usage: kaggle [-h] [-v] [-W]
              {competitions,c,datasets,d,kernels,k,models,m,files,f,benchmarks,b,config,auth} ...
kaggle: error: argument command: invalid choice: 'dataset' (choose from 'competitions', 'c', 'datasets', 'd', 'kernels', 'k', 'models', 'm', 'files', 'f', 'benchmarks', 'b', 'config', 'auth')


In [ ]:
import os
import random
import shutil
os.environ['KAGGLE_USERNAME'] = "niharpun"
os.environ['KAGGLE_KEY'] = "KGAT_5fc66ddbe6e6e6edd3bd0a7fc880f157"
!kaggle datasets list
!kaggle datasets download -d raghavdharwal/vehicle-orientation-dataset-part-1
!unzip -q vehicle-orientation-dataset-part-1.zip -d vehicle_data


ref                                                             title                                                     size  lastUpdated                 downloadCount  voteCount  usabilityRating  
--------------------------------------------------------------  --------------------------------------------------  ----------  --------------------------  -------------  ---------  ---------------  
datascikhan/e-commerce-sales-and-customer-analytics             E-Commerce Sales Analytics Dataset                    29815424  2026-08-25 08:00:46.013000           8276        174                1  
srisyra02/online-food-ordering-dataset                          Online Food Ordering Dataset                              3648  2026-08-23 15:26:25.483000           5620         90                1  
mobeenfatimah/student-exam-performance-and-success-dataset      Student Exam Performance &  Success Dataset            6004839  2026-08-19 07:30:23.257000           4704         83                1  


In [ ]:
import random
import shutil
from ultralytics import YOLO

source_path = "/content/vehicle_data/vehicle-orientation-1"
new_dataset = "/content/YOLO_format_dataset"

if os.path.exists(new_dataset):
    shutil.rmtree(new_dataset)

for folders in ["images/train", "images/val", "labels/train" , "labels/val"]:
  os.makedirs(os.path.join(new_dataset, folders), exist_ok=True)

if "vehicle-orientation-1" in os.listdir(source_path):
    source_path = os.path.join(source_path, 'vehicle-orientation-1')

all_images = [f for f in os.listdir(source_path) if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
valid_pairs = []

for image_files in all_images:
    base_name = os.path.splitext(image_files)[0]
    txt_file = f"{base_name}.txt"
    if os.path.exists(os.path.join(source_path, txt_file)):
        valid_pairs.append((image_files, txt_file))

random.seed(42)
valid_pairs = random.sample(valid_pairs, 1000)

random.shuffle(valid_pairs)
split = int(len(valid_pairs) * 0.8)

train_pairs = valid_pairs[:split]
val_pairs = valid_pairs[split:]

def copy_dataset_split(pairs_list, split_name):
    for img, txt in pairs_list:
        shutil.copy(os.path.join(source_path, img), os.path.join(new_dataset, 'images', split_name, img))
        shutil.copy(os.path.join(source_path, txt), os.path.join(new_dataset, 'labels', split_name, txt))

copy_dataset_split(train_pairs, 'train')
copy_dataset_split(val_pairs, 'val')

yaml_content = f"""
path: {new_dataset}
train: images/train
val: images/val

nc: 15
names:
  0: car_back
  1: car_side
  2: car_front
  3: bus_back
  4: bus_side
  5: bus_front
  6: truck_back
  7: truck_side
  8: truck_front
  9: motorcycle_back
  10: motorcycle_side
  11: motorcycle_front
  12: bicycle_back
  13: bicycle_side
  14: bicycle_front
"""

with open(os.path.join(new_dataset, 'data.yaml'), 'w') as f:
    f.write(yaml_content.strip())


In [2]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
model = YOLO("yolov8n.pt")
resuls = model.train(
    data = "/content/YOLO_format_dataset/data.yaml",
    epochs = 15,
    imgsz = 640,
    device = 0,
    plots = True,
)

Ultralytics 8.4.154 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/YOLO_format_dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=None, opset=None, 

In [ ]:
test = model("/content/UCMUWN6QSQMBXLRVYUQT.jpg", save = True)
for result in test:
    boxes = result.boxes  # Bounding boxes object
    for box in boxes:
        # Get coordinates, confidence, and class label index
        confidence = box.conf[0].item()
        class_id = int(box.cls[0].item())

        print(f"Class: {class_id}, Conf: {confidence:.2f}")


image 1/1 /content/UCMUWN6QSQMBXLRVYUQT.jpg: 384x640 2 car_backs, 1 truck_back, 11.7ms
Speed: 3.2ms preprocess, 11.7ms inference, 1.9ms postprocess per image at shape (1, 3, 384, 640)
Results saved to /content/runs/detect/predict
Class: 0, Conf: 0.97
Class: 0, Conf: 0.97
Class: 6, Conf: 0.67


In [ ]:
!mkdir -p /content/drive/MyDrive/YOLO_Training_Results
!cp -r /content/runs/detect/train /content/drive/MyDrive/YOLO_Training_Results/


In [4]:
YOLO("/content/drive/MyDrive/YOLO_Training_Results/train/weights/best.pt")

YOLO(
  (model): DetectionModel(
    (model): Sequential(
      (0): Conv(
        (conv): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (1): Conv(
        (conv): Conv2d(16, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
        (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
        (act): SiLU(inplace=True)
      )
      (2): C2f(
        (cv1): Conv(
          (conv): Conv2d(32, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (act): SiLU(inplace=True)
        )
        (cv2): Conv(
          (conv): Conv2d(48, 32, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn): BatchNorm2d(32, eps=0.001, momentum=0.03, affine=True, track_running_s

In [5]:
!pip install gradio


In [8]:
import gradio as gr
from ultralytics import YOLO
from PIL import Image

model = YOLO("/content/drive/MyDrive/YOLO_Training_Results/train/weights/best.pt")

def predict_image(img):
    results = model(img)
    annotated_img = results[0].plot()
    return Image.fromarray(annotated_img[..., ::-1])

interface = gr.Interface(
    fn=predict_image,
    inputs=gr.Image(type="pil"),
    outputs=gr.Image(type="pil"),
    title="YOLO Vehicle orienations :)"
)

interface.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://34219ff815090e8d06.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
